# EduGuard AI — Student Risk Detection Model

This notebook builds the **EduGuard AI risk-detection ML model** from the enriched student dataset.

### Pipeline
`CSV → preprocessing → feature selection → model comparison → evaluation → risk probability → predictions → model export`

> **Important:** The enriched behavioral/risk data in this project is synthetic/approximate for development and demonstration. Do not treat the reported metrics as real-world student-risk performance.


## 1. Install / import libraries

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42


## 2. Upload the enriched dataset

Upload the `student_dataset_enriched.csv` file generated for EduGuard.

In [ ]:
uploaded = files.upload()

DATASET_PATH = next(iter(uploaded.keys()))
df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())


## 3. Inspect the dataset

In [ ]:
print("Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nTarget distribution:")
display(df["At_Risk"].value_counts())
display(df["At_Risk"].value_counts(normalize=True).rename("proportion"))


## 4. Prepare ML features

We intentionally exclude:

- Personally identifying information
- Raw feedback text
- The sentiment label from the structured model
- `Risk_Score`, `Risk_Level`, and `At_Risk` because these are target-derived fields
- `Grade` and `School Name` from this first structured model

This prevents direct target leakage.

In [ ]:
TARGET = "At_Risk"

EXCLUDE = [
    "Student_Names",
    "Phone_No.",
    "Student Address",
    "Roll No.",
    "Comment",
    "Weekly_Feedback",
    "Sentiment_Label",
    "At_Risk",
    "Risk_Level",
    "Risk_Score",
    "Grade",
    "School Name"
]

FEATURES = [c for c in df.columns if c not in EXCLUDE]

X = df[FEATURES].apply(pd.to_numeric, errors="coerce")
y = df[TARGET].astype(int)

print(f"Number of ML features: {len(FEATURES)}")
print(FEATURES)
print("\nTarget counts:")
print(y.value_counts())


## 5. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 6. Train multiple models

We compare:

1. Logistic Regression — strong interpretable baseline
2. Random Forest — nonlinear tree ensemble
3. Gradient Boosting — boosted tree model

For an early-warning system, **recall** is especially important because missing an at-risk student is costly.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(
            n_estimators=180,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE
        ))
    ])
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })

    fitted_models[name] = model

results_df = (
    pd.DataFrame(results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "ROC_AUC": "{:.4f}"
}))


## 7. Select the model

The default selection criterion is **F1**, balancing precision and recall.

If your project prioritizes catching as many genuinely at-risk students as possible, you can instead select based on **Recall**.

In [ ]:
BEST_MODEL_NAME = results_df.iloc[0]["Model"]
BEST_MODEL = fitted_models[BEST_MODEL_NAME]

print("Selected model:", BEST_MODEL_NAME)


## 8. Detailed evaluation

In [ ]:
best_pred = BEST_MODEL.predict(X_test)
best_prob = BEST_MODEL.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    best_pred,
    target_names=["Not At Risk", "At Risk"],
    digits=4
))

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Not At Risk", "At Risk"],
    yticklabels=["Not At Risk", "At Risk"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {BEST_MODEL_NAME}")
plt.show()


## 9. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, best_prob)
precision, recall, _ = precision_recall_curve(y_test, best_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, best_prob):.3f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
plt.show()


## 10. Feature importance / model explanation

For tree-based models, feature importance is available directly. For Logistic Regression, the absolute coefficient magnitude can be used as a simple indicator of influence.

This is intended to support dashboard explanations such as:

> Low attendance, declining scores, and high late-submission behavior contributed to the predicted risk.



In [ ]:
model_step = BEST_MODEL.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    importance = model_step.feature_importances_
elif hasattr(model_step, "coef_"):
    importance = np.abs(model_step.coef_[0])
else:
    importance = None

if importance is not None:
    importance_df = pd.DataFrame({
        "Feature": FEATURES,
        "Importance": importance
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(15))

    plt.figure(figsize=(9, 6))
    top = importance_df.head(15).sort_values("Importance")
    plt.barh(top["Feature"], top["Importance"])
    plt.xlabel("Importance")
    plt.title(f"Top Risk-Prediction Features — {BEST_MODEL_NAME}")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance is not directly available for this model.")


## 11. Generate risk predictions for every student

Risk probability:

- `< 0.40` → Low
- `0.40 to < 0.65` → Medium
- `>= 0.65` → High

The binary `At_Risk` prediction still uses the model's classification threshold.

In [ ]:
all_prob = BEST_MODEL.predict_proba(X)[:, 1]
all_pred = (all_prob >= 0.50).astype(int)

predictions = df[["Student_Names", "Grade"]].copy()

predictions["Predicted_Risk_Probability"] = np.round(all_prob, 4)
predictions["Predicted_At_Risk"] = all_pred
predictions["Predicted_Risk_Level"] = np.select(
    [
        all_prob >= 0.65,
        all_prob >= 0.40
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

display(predictions.head(20))

print("\nRisk-level distribution:")
display(predictions["Predicted_Risk_Level"].value_counts())


## 12. Export predictions and trained model

In [ ]:
PREDICTIONS_FILE = "eduguard_risk_predictions.csv"
MODEL_FILE = "eduguard_risk_model.joblib"
METRICS_FILE = "eduguard_model_metrics.csv"

predictions.to_csv(PREDICTIONS_FILE, index=False)
results_df.to_csv(METRICS_FILE, index=False)
joblib.dump(BEST_MODEL, MODEL_FILE)

print("Saved:")
print(PREDICTIONS_FILE)
print(MODEL_FILE)
print(METRICS_FILE)


## 13. Test the model on one student

Pass a dictionary containing the same 40 model features.

In your FastAPI backend, this same logic can be placed behind `/predict`.

In [ ]:
def predict_student(student_features: dict):
    row = pd.DataFrame([student_features])

    # Keep exactly the training feature order.
    row = row.reindex(columns=FEATURES)

    risk_probability = float(BEST_MODEL.predict_proba(row)[0, 1])

    if risk_probability >= 0.65:
        risk_level = "High"
    elif risk_probability >= 0.40:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    return {
        "risk_probability": round(risk_probability, 4),
        "risk_level": risk_level,
        "at_risk": bool(risk_probability >= 0.50)
    }

# Example: use the first test student
example_features = X_test.iloc[0].to_dict()
predict_student(example_features)


## 14. Download the generated artifacts

In [ ]:
files.download(PREDICTIONS_FILE)
files.download(MODEL_FILE)
files.download(METRICS_FILE)


# Next step: EduGuard API

The saved `eduguard_risk_model.joblib` can be loaded in FastAPI:

```python
model = joblib.load("eduguard_risk_model.joblib")
probability = model.predict_proba(features)[0, 1]
```

Then your React dashboard can display:

- Risk probability
- Low / Medium / High risk
- Top contributing factors
- Student trends
- Recommended intervention

**Note:** The current notebook deliberately treats feedback sentiment as a separate future NLP component, matching the EduGuard architecture. The structured risk model does not use raw feedback text.